# BELTA IL (Impermanent Loss) 분석 노트북

Uniswap V2/V3 비영구적 손실(IL) 계산 및 시각화

### 목차
1. IL 기본 공식 (V2)
2. V3 집중 유동성 IL 보정
3. 가격 변동별 IL 시뮬레이션
4. Range 폭에 따른 IL 증폭 분석
5. LP 세그먼트별 IL 비교
6. BELTA 헤지 효과 분석
7. 실제 ETH 가격 기반 에포크별 IL

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats as sst

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

# BELTA 프로토콜 파라미터
COVERAGE_CAP = 0.35   # IL의 최대 35% 보상
LP_FEE_MID = 0.224    # ETH/USDC 0.05% 풀 연간 수수료율
LP_RATE = 0.12        # 프리미엄율 12%
EPOCH_DAYS = 7        # 에포크 주기

print('Setup complete.')

## 1. IL 기본 공식 (Uniswap V2)

$$\text{IL}_{V2} = \frac{2\sqrt{r}}{1 + r} - 1$$

여기서 $r = P_{\text{end}} / P_{\text{start}}$ (가격 비율)

- IL은 **방향 무관**, 가격 변동 **폭**에만 비례
- $r = 1$ (가격 불변) → IL = 0
- $r = 0.5$ (50% 하락) → IL ≈ 5.72%
- $r = 2.0$ (100% 상승) → IL ≈ 5.72%

In [ ]:
def il_v2(r):
    """Uniswap V2 IL: r = price_end / price_start"""
    if r <= 0:
        return 0.0
    return abs(2 * np.sqrt(r) / (1 + r) - 1)

# 벡터화 버전
def il_v2_vec(r):
    r = np.asarray(r, dtype=float)
    r = np.clip(r, 1e-10, None)
    return np.abs(2 * np.sqrt(r) / (1 + r) - 1)

# 가격 변동 범위: -90% ~ +500%
price_changes = np.linspace(0.1, 6.0, 500)
il_values = il_v2_vec(price_changes) * 100  # 퍼센트

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(price_changes, il_values, color='#c8a96e', linewidth=2.5)
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.5, label='No price change')

# 주요 포인트 표시
key_points = [0.5, 0.75, 1.25, 1.5, 2.0, 3.0]
for r in key_points:
    il = il_v2(r) * 100
    pct = (r - 1) * 100
    ax.plot(r, il, 'o', color='#e74c3c', markersize=8)
    ax.annotate(f'{pct:+.0f}% → IL {il:.1f}%',
                xy=(r, il), xytext=(10, 10),
                textcoords='offset points', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.7))

ax.set_xlabel('Price Ratio (P_end / P_start)', fontsize=12)
ax.set_ylabel('Impermanent Loss (%)', fontsize=12)
ax.set_title('Uniswap V2 Impermanent Loss vs Price Change', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 5)
ax.set_ylim(0, 35)
plt.tight_layout()
plt.show()

## 2. V3 집중 유동성 IL 보정

V3에서는 유동성이 특정 가격 범위 $[P_a, P_b]$에 집중되므로 IL이 증폭됨:

$$\text{IL}_{V3} = \text{IL}_{V2} \times \frac{1}{1 - \sqrt{P_a / P_b}}$$

- 범위가 좁을수록 → $P_a \approx P_b$ → 보정계수 ↑ → IL 증폭
- Full range ($P_a \to 0, P_b \to \infty$) → 보정계수 = 1 (V2와 동일)

In [ ]:
def il_v3_correction(pa, pb):
    """V3 IL correction factor given range [pa, pb]"""
    ratio = np.sqrt(pa / pb)
    return 1.0 / (1.0 - ratio)

def il_v3(r, pa, pb):
    """V3 IL = V2 IL * correction factor"""
    return il_v2(r) * il_v3_correction(pa, pb)

# 다양한 range 폭에 대한 보정계수 계산
# range_width = (pb - pa) / current_price (현재가 대비 range 폭 비율)
current_price = 3000  # ETH/USDC

range_widths = np.linspace(0.02, 2.0, 200)  # 2% ~ 200%
corrections = []
for w in range_widths:
    pa = current_price * (1 - w/2)
    pb = current_price * (1 + w/2)
    pa = max(pa, 1)  # 음수 방지
    corrections.append(il_v3_correction(pa, pb))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 좌: 보정계수 vs range 폭
ax1.plot(range_widths * 100, corrections, color='#3498db', linewidth=2.5)
ax1.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='V2 (Full Range)')

# LP 세그먼트 표시
segments = [
    ('Institutional (narrow)', 0.10, '#c8a96e', 2.8),
    ('Active Retail (medium)', 0.30, '#3498db', 1.5),
    ('Passive Retail (wide)', 0.80, '#e74c3c', 0.7),
]
for label, w, color, mult in segments:
    pa = current_price * (1 - w/2)
    pb = current_price * (1 + w/2)
    corr = il_v3_correction(pa, pb)
    ax1.plot(w * 100, corr, 'o', color=color, markersize=12, zorder=5)
    ax1.annotate(f'{label}\n{corr:.1f}x',
                xy=(w * 100, corr), xytext=(15, 5),
                textcoords='offset points', fontsize=9,
                bbox=dict(boxstyle='round', facecolor=color, alpha=0.2))

ax1.set_xlabel('Range Width (% of current price)', fontsize=12)
ax1.set_ylabel('IL Correction Factor (multiplier)', fontsize=12)
ax1.set_title('V3 IL Amplification by Range Width', fontsize=13, fontweight='bold')
ax1.set_xlim(0, 100)
ax1.set_ylim(0, 25)
ax1.legend(fontsize=10)

# 우: 세그먼트별 IL 비교 (다양한 가격 변동)
price_ratios = np.linspace(0.5, 2.0, 200)

il_full = il_v2_vec(price_ratios) * 100
ax2.plot(price_ratios, il_full, color='gray', linewidth=1.5, linestyle='--', label='V2 Full Range (1.0x)')

for label, w, color, mult in segments:
    il_seg = il_v2_vec(price_ratios) * mult * 100
    ax2.plot(price_ratios, il_seg, color=color, linewidth=2, label=f'{label} ({mult}x)')

ax2.set_xlabel('Price Ratio (P_end / P_start)', fontsize=12)
ax2.set_ylabel('Impermanent Loss (%)', fontsize=12)
ax2.set_title('IL by LP Segment (V3 Correction Applied)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_ylim(0, 40)

plt.tight_layout()
plt.show()

## 3. 가격 변동 시나리오별 IL 시뮬레이션

실제 ETH 가격 시나리오에서의 IL 크기를 분석

In [ ]:
# 실제 역사적 이벤트 기반 시나리오
scenarios = {
    'Normal week':          {'change': 0.03,  'desc': 'Typical 7-day move'},
    'Moderate volatility':  {'change': 0.08,  'desc': 'Moderate weekly swing'},
    'High volatility':      {'change': 0.15,  'desc': 'Significant move'},
    'COVID crash (2020.3)': {'change': -0.32, 'desc': 'ETH $230→$88'},
    'LUNA crash (2022.5)':  {'change': -0.35, 'desc': 'UST depeg contagion'},
    'FTX crash (2022.11)':  {'change': -0.25, 'desc': 'Exchange collapse'},
    'Aug 2024 dump':        {'change': -0.18, 'desc': 'Carry trade unwind'},
    'Bull run pump':        {'change': 0.40,  'desc': 'Strong upward move'},
}

seg_mults = {'Institutional (2.8x)': 2.8, 'Active Retail (1.5x)': 1.5, 'Passive Retail (0.7x)': 0.7}

rows = []
for scenario, info in scenarios.items():
    r = 1 + info['change']
    base_il = il_v2(r)
    row = {
        'Scenario': scenario,
        'Price Change': f"{info['change']:+.0%}",
        'V2 IL': f"{base_il:.2%}",
    }
    for seg_name, mult in seg_mults.items():
        seg_il = base_il * mult
        hedged = seg_il * (1 - COVERAGE_CAP)
        row[seg_name] = f"{seg_il:.2%}"
        row[f'{seg_name} (hedged)'] = f"{hedged:.2%}"
    rows.append(row)

df_scenarios = pd.DataFrame(rows)
display_cols = ['Scenario', 'Price Change', 'V2 IL',
                'Institutional (2.8x)', 'Active Retail (1.5x)', 'Passive Retail (0.7x)']
df_scenarios[display_cols].style.set_caption('Scenario-based IL Analysis by LP Segment')

In [ ]:
# 시나리오별 IL 시각화 (세그먼트 비교)
fig, ax = plt.subplots(figsize=(14, 7))

scenario_names = list(scenarios.keys())
x = np.arange(len(scenario_names))
width = 0.2

colors = {'Institutional (2.8x)': '#c8a96e', 'Active Retail (1.5x)': '#3498db', 'Passive Retail (0.7x)': '#e74c3c'}

for i, (seg_name, mult) in enumerate(seg_mults.items()):
    ils = [il_v2(1 + s['change']) * mult * 100 for s in scenarios.values()]
    hedged = [il * (1 - COVERAGE_CAP) for il in ils]
    
    bars = ax.bar(x + i * width, ils, width, label=seg_name,
                  color=colors[seg_name], alpha=0.8)
    # 헤지 후 IL (어두운 색 오버레이)
    ax.bar(x + i * width, hedged, width,
           color=colors[seg_name], alpha=0.4, hatch='//',
           label=f'{seg_name} (after BELTA hedge)' if i == 0 else None)

ax.set_xlabel('Scenario', fontsize=12)
ax.set_ylabel('Impermanent Loss (%)', fontsize=12)
ax.set_title('IL by Scenario & LP Segment\n(hatched = after BELTA 35% coverage)', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(scenario_names, rotation=35, ha='right', fontsize=9)
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

## 4. Range 폭에 따른 IL 증폭 히트맵

가격 변동(x축) × Range 폭(y축) → IL 크기(색상)

In [ ]:
# 히트맵: 가격 변동 vs Range 폭 → IL
price_changes_pct = np.linspace(-40, 40, 81)  # -40% ~ +40%
range_widths_pct = np.linspace(5, 100, 48)     # 5% ~ 100%

il_matrix = np.zeros((len(range_widths_pct), len(price_changes_pct)))

for i, rw in enumerate(range_widths_pct):
    pa = current_price * (1 - rw / 200)
    pb = current_price * (1 + rw / 200)
    corr = il_v3_correction(max(pa, 1), pb)
    for j, pc in enumerate(price_changes_pct):
        r = 1 + pc / 100
        il_matrix[i, j] = il_v2(r) * corr * 100

# 45% 상한 클리핑 (BELTA 프로토콜 상한)
il_matrix = np.clip(il_matrix, 0, 45)

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(il_matrix, aspect='auto', origin='lower',
               extent=[price_changes_pct[0], price_changes_pct[-1],
                       range_widths_pct[0], range_widths_pct[-1]],
               cmap='YlOrRd')

cbar = plt.colorbar(im, ax=ax, label='Impermanent Loss (%)')

# LP 세그먼트 위치 표시
seg_ranges = {'Institutional': 10, 'Active Retail': 30, 'Passive Retail': 80}
for label, rw in seg_ranges.items():
    ax.axhline(rw, color='white', linestyle='--', alpha=0.7, linewidth=1.5)
    ax.text(price_changes_pct[-1] - 2, rw + 1.5, label,
            color='white', fontsize=9, fontweight='bold', ha='right',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))

ax.set_xlabel('Price Change (%)', fontsize=12)
ax.set_ylabel('Range Width (% of price)', fontsize=12)
ax.set_title('V3 IL Heatmap: Price Change vs Range Width\n(Narrower range = higher IL amplification)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. LP 세그먼트별 IL 비교 (수수료 포함 순손익)

IL만 보면 패시브 리테일이 유리해 보이지만, 좁은 범위 LP는 수수료 수익이 훨씬 높음.

**실질 수익 = Fee APY × (range correction) - IL**

In [ ]:
# 세그먼트별 7일 에포크 순손익 분석
# Fee도 range에 비례해 집중됨 (좁은 범위 = 높은 수수료 밀도)

seg_configs = {
    'Institutional (narrow)': {
        'il_mult': 2.8,
        'fee_mult': 3.5,   # narrow range → fee 집중 3.5x
        'color': '#c8a96e',
    },
    'Active Retail (medium)': {
        'il_mult': 1.5,
        'fee_mult': 1.8,
        'color': '#3498db',
    },
    'Passive Retail (wide)': {
        'il_mult': 0.7,
        'fee_mult': 0.6,   # wide range → fee 희석
        'color': '#e74c3c',
    },
}

weekly_fee_base = LP_FEE_MID / 52  # 주간 기본 수수료율

price_changes_sim = np.linspace(-0.30, 0.30, 200)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for idx, (seg_name, cfg) in enumerate(seg_configs.items()):
    ax = axes[idx]
    
    ils = [il_v2(1 + pc) * cfg['il_mult'] * 100 for pc in price_changes_sim]
    fees = [weekly_fee_base * cfg['fee_mult'] * 100] * len(price_changes_sim)  # 수수료는 가격 변동 무관
    net = [f - il for f, il in zip(fees, ils)]
    
    # BELTA 헤지 후 순손익
    ils_hedged = [il * (1 - COVERAGE_CAP) for il in ils]
    net_hedged = [f - il for f, il in zip(fees, ils_hedged)]
    
    ax.fill_between(price_changes_sim * 100, net, 0,
                    where=[n >= 0 for n in net], color=cfg['color'], alpha=0.3, label='Profit')
    ax.fill_between(price_changes_sim * 100, net, 0,
                    where=[n < 0 for n in net], color='red', alpha=0.2, label='Loss')
    ax.plot(price_changes_sim * 100, net, color=cfg['color'], linewidth=2, label='Net P&L (no hedge)')
    ax.plot(price_changes_sim * 100, net_hedged, color='green', linewidth=2,
            linestyle='--', label='Net P&L (BELTA hedged)')
    ax.axhline(0, color='black', linewidth=0.8)
    
    ax.set_xlabel('Weekly Price Change (%)', fontsize=10)
    ax.set_title(seg_name, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Weekly Net P&L (%)', fontsize=11)
fig.suptitle('Weekly Net P&L by LP Segment (Fee - IL)\nDashed = with BELTA 35% IL hedge',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. BELTA 헤지 효과 분석

BELTA가 IL의 35%를 커버할 때의 실질 효과:
- Coverage Cap: 35%
- 프리미엄: Fee 수익의 12%
- 순 헤지 효과 = Coverage - Premium

In [ ]:
# BELTA 헤지 손익분기점 분석
price_changes_belta = np.linspace(-0.50, 0.50, 500)

# 기관 LP 기준
mult = 2.8
weekly_fee = weekly_fee_base * 3.5  # narrow range fee
premium = weekly_fee * LP_RATE       # BELTA 프리미엄 비용

no_hedge_pnl = []
hedged_pnl = []
hedge_benefit = []

for pc in price_changes_belta:
    r = 1 + pc
    il = il_v2(r) * mult
    
    # 헤지 없이
    pnl_no = weekly_fee - il
    no_hedge_pnl.append(pnl_no * 100)
    
    # BELTA 헤지
    il_after = il * (1 - COVERAGE_CAP)
    pnl_hedged = weekly_fee - premium - il_after
    hedged_pnl.append(pnl_hedged * 100)
    
    # 순 혜택
    hedge_benefit.append((pnl_hedged - pnl_no) * 100)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# 상단: P&L 비교
ax1.plot(price_changes_belta * 100, no_hedge_pnl, color='#e74c3c',
         linewidth=2, label='Without BELTA hedge')
ax1.plot(price_changes_belta * 100, hedged_pnl, color='#27ae60',
         linewidth=2, label='With BELTA hedge (35% coverage)')
ax1.axhline(0, color='black', linewidth=0.8)
ax1.fill_between(price_changes_belta * 100, no_hedge_pnl, hedged_pnl,
                 alpha=0.15, color='green', label='Hedge benefit zone')

ax1.set_xlabel('Weekly Price Change (%)', fontsize=11)
ax1.set_ylabel('Weekly P&L (%)', fontsize=11)
ax1.set_title('Institutional LP: P&L With vs Without BELTA Hedge',
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)

# 하단: 순 헤지 혜택
ax2.fill_between(price_changes_belta * 100, hedge_benefit, 0,
                 where=[b >= 0 for b in hedge_benefit],
                 color='green', alpha=0.3, label='Hedge beneficial')
ax2.fill_between(price_changes_belta * 100, hedge_benefit, 0,
                 where=[b < 0 for b in hedge_benefit],
                 color='red', alpha=0.3, label='Premium > benefit')
ax2.plot(price_changes_belta * 100, hedge_benefit, color='#2c3e50', linewidth=2)
ax2.axhline(0, color='black', linewidth=0.8)

# BEP 찾기
for i in range(1, len(hedge_benefit)):
    if hedge_benefit[i-1] * hedge_benefit[i] < 0:
        bep = price_changes_belta[i] * 100
        ax2.axvline(bep, color='orange', linestyle='--', alpha=0.7)
        ax2.annotate(f'BEP: {bep:.1f}%', xy=(bep, 0), xytext=(bep + 3, 0.02),
                    fontsize=10, color='orange', fontweight='bold')

ax2.set_xlabel('Weekly Price Change (%)', fontsize=11)
ax2.set_ylabel('Net Hedge Benefit (%)', fontsize=11)
ax2.set_title('Net Benefit of BELTA Hedge\n(positive = hedge is worth the premium)',
              fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 7. 실제 ETH 가격 기반 에포크별 IL

2020~2025 시뮬레이션 가격 데이터 기반 에포크별 IL 추이

In [ ]:
# 가격 시뮬레이션 (backtest v4.4.0과 동일)
np.random.seed(42)

WAYPOINTS = [
    ('2020-01-01',  130), ('2020-03-13',   88), ('2021-01-01',  730),
    ('2021-05-12', 4150), ('2021-11-10', 4800), ('2022-06-18',  900),
    ('2022-11-09', 1150), ('2023-06-01', 1900), ('2024-03-12', 4000),
    ('2025-03-01', 2200),
]
dates_dt = pd.date_range('2020-01-01', '2025-03-01', freq='D')
N = len(dates_dt)
wp_d = [pd.Timestamp(w[0]) for w in WAYPOINTS]
wp_p = [w[1] for w in WAYPOINTS]

base = np.zeros(N)
for i, d in enumerate(dates_dt):
    idx = max(j for j, wd in enumerate(wp_d) if wd <= d)
    if idx < len(wp_d) - 1:
        t0, t1 = wp_d[idx], wp_d[idx+1]
        p0, p1 = wp_p[idx], wp_p[idx+1]
        r = (d - t0).days / max((t1 - t0).days, 1)
        base[i] = p0 * (p1 / p0) ** r
    else:
        base[i] = wp_p[-1]

df_t = 4
scale = np.sqrt(df_t / (df_t - 2))
prices = base.copy()
for i in range(1, N):
    noise = sst.t.rvs(df=df_t) * 0.045 / scale
    jump = np.random.normal(-0.05, 0.12) if np.random.random() < 0.015 else 0.0
    total = np.clip(noise + jump, -0.15, 0.15)
    prices[i] = (base[i] * (1 + total * 0.3) + prices[i-1] * (1 + total * 0.7)) / 2
    prices[i] = max(prices[i], 1.0)

# 에포크별 계산
epoch_dates = dates_dt[::7]
epoch_prices = prices[::7]
n_epochs = len(epoch_prices) - 1

weekly_returns = np.diff(epoch_prices) / epoch_prices[:-1]

# 세그먼트별 IL 계산
seg_il = {}
for seg_name, mult in [('Institutional', 2.8), ('Active Retail', 1.5), ('Passive Retail', 0.7)]:
    ils = [min(il_v2(1 + ret) * mult, 0.45) * 100 for ret in weekly_returns]
    seg_il[seg_name] = ils

print(f'Total epochs: {n_epochs}')
print(f'Date range: {epoch_dates[0].strftime("%Y-%m-%d")} ~ {epoch_dates[-1].strftime("%Y-%m-%d")}')

In [ ]:
# 에포크별 IL 추이 + ETH 가격
fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

# (1) ETH 가격
ax1 = axes[0]
ax1.plot(epoch_dates[:n_epochs], epoch_prices[:n_epochs], color='#c8a96e', linewidth=1.5)
ax1.set_ylabel('ETH Price ($)', fontsize=11)
ax1.set_title('ETH Price & Epoch-level IL Analysis (2020-2025)', fontsize=14, fontweight='bold')

# 주요 이벤트 표시
events = [
    ('2020-03-13', 'COVID'),
    ('2021-05-12', 'ATH $4.1K'),
    ('2022-05-10', 'LUNA'),
    ('2022-11-09', 'FTX'),
    ('2024-08-05', 'Aug dump'),
]
for date_str, label in events:
    try:
        evt_date = pd.Timestamp(date_str)
        if evt_date in epoch_dates[:n_epochs].values or True:
            nearest = epoch_dates[:n_epochs][np.argmin(np.abs(epoch_dates[:n_epochs] - evt_date))]
            price_at = epoch_prices[np.argmin(np.abs(epoch_dates[:n_epochs] - evt_date))]
            ax1.annotate(label, xy=(nearest, price_at),
                        xytext=(0, 20), textcoords='offset points',
                        fontsize=8, ha='center',
                        arrowprops=dict(arrowstyle='->', color='red', alpha=0.7),
                        bbox=dict(boxstyle='round', facecolor='lightyellow'))
    except:
        pass
ax1.grid(True, alpha=0.3)

# (2) 세그먼트별 IL
ax2 = axes[1]
colors = {'Institutional': '#c8a96e', 'Active Retail': '#3498db', 'Passive Retail': '#e74c3c'}
for seg_name, ils in seg_il.items():
    ax2.plot(epoch_dates[:n_epochs], ils[:n_epochs], color=colors[seg_name],
             linewidth=1, alpha=0.7, label=seg_name)

ax2.set_ylabel('Epoch IL (%)', fontsize=11)
ax2.set_title('Per-Epoch IL by LP Segment', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# (3) 누적 IL (BELTA 헤지 전후 비교) - 기관 LP 기준
ax3 = axes[2]
inst_il = np.array(seg_il['Institutional'][:n_epochs])
cum_il = np.cumsum(inst_il)
cum_il_hedged = np.cumsum(inst_il * (1 - COVERAGE_CAP))

ax3.fill_between(epoch_dates[:n_epochs], cum_il, cum_il_hedged,
                 alpha=0.3, color='green', label='BELTA coverage')
ax3.plot(epoch_dates[:n_epochs], cum_il, color='#e74c3c',
         linewidth=1.5, label='Cumulative IL (no hedge)')
ax3.plot(epoch_dates[:n_epochs], cum_il_hedged, color='#27ae60',
         linewidth=1.5, label='Cumulative IL (BELTA hedged)')

ax3.set_ylabel('Cumulative IL (%)', fontsize=11)
ax3.set_xlabel('Date', fontsize=11)
ax3.set_title('Institutional LP: Cumulative IL with vs without BELTA (35% coverage)', fontsize=12)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 요약 통계
print('\n=== IL Summary Statistics (per epoch) ===')
for seg_name, ils in seg_il.items():
    arr = np.array(ils[:n_epochs])
    print(f'\n{seg_name}:')
    print(f'  Mean IL:   {arr.mean():.3f}%')
    print(f'  Median IL: {np.median(arr):.3f}%')
    print(f'  Max IL:    {arr.max():.3f}%')
    print(f'  Std IL:    {arr.std():.3f}%')
    print(f'  Total cumulative IL: {arr.sum():.1f}%')

## Summary

### Key Findings

1. **V3 IL Amplification**: Narrow range LPs (institutional) face 2.8x more IL than V2 full range
2. **BELTA Coverage Effect**: 35% coverage significantly reduces tail risk during black swan events
3. **Break-even Analysis**: BELTA hedge becomes net positive when weekly price change exceeds ~3-5%
4. **Segment Trade-off**: Institutional LPs have higher IL but also much higher fee income from concentrated liquidity
5. **Cumulative Impact**: Over 5 years, BELTA hedge saves ~35% of cumulative IL for institutional LPs